[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-biology/AdvancesMSDataProcessing/blob/master/SpectralDataPreprocessingTreatment/basic_mass_spectrum_preprocessing_peak_detection.ipynb)


# Basic mass-spectrum preprocessing and peak detection

A reproducible example of the workflow shown in Figure 2 of the BMC Bioinformatics paper: **raw spectrum → smoothing → baseline correction → peak detection**.

The implementation uses the same core ideas as the two supplied Python sources: Savitzky–Golay smoothing and `scipy.signal.find_peaks`, with polynomial baseline subtraction. The published figure is used as a conceptual reference; this notebook generates a new synthetic spectrum rather than copying the published image.

Sources:
- basic spectral manipulation: https://github.com/computational-chemical-biology/basic_data_manipulation/blob/master/notebooks/basic_spectral_manipulation.ipynb
- bacterial processing methods: https://github.com/chrisp33/Python-For-RnD/blob/main/Bacteria_Proc_Methods.py
- Figure 2: https://link.springer.com/article/10.1186/1471-2105-10-4/figures/2


## 1. Install dependencies

In [ ]:
%pip install -q numpy pandas scipy matplotlib pyopenms

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks


## 3. Basic preprocessing functions

These are single-spectrum versions of the `SmoothData`, `BaselineCorrect`, and `GetPeaks` ideas in `Bacteria_Proc_Methods.py`.


In [ ]:
def smooth_spectrum(intensity, window_length=41, polyorder=3):
    if window_length % 2 == 0:
        window_length += 1
    return savgol_filter(intensity, window_length, polyorder)

def polynomial_baseline(intensity, degree=3, edge_fraction=0.10):
    n = len(intensity)
    edge_n = max(5, int(n * edge_fraction))
    x = np.arange(n, dtype=float)
    idx = np.r_[np.arange(edge_n), np.arange(n-edge_n, n)]
    coef = np.polyfit(x[idx], intensity[idx], degree)
    return np.polyval(coef, x)

def baseline_correct(intensity, degree=3, edge_fraction=0.10):
    baseline = polynomial_baseline(intensity, degree, edge_fraction)
    corrected = intensity - baseline
    corrected[corrected < 0] = 0
    return corrected, baseline

def detect_peaks(mz, intensity, prominence=120, distance=40):
    return find_peaks(intensity, prominence=prominence, distance=distance)


## 4. Generate a synthetic mass spectrum

The synthetic signal has the main characteristics of the reference: a strong low-m/z background, broad baseline, narrow analytical peaks, and noise.


In [ ]:
rng = np.random.default_rng(42)
mz = np.linspace(2000, 50000, 12000)

background = 1050 + 6000*np.exp(-(mz-2500)/4500) + 450*np.exp(-(mz-5000)/17000)
noise = rng.normal(0, 55, mz.size) + 35*np.sin(mz/700) + 20*np.sin(mz/125)
raw = background + noise

peak_specs = [
    (3600,4200,55),(6200,3200,45),(7800,2800,50),(9200,3900,45),
    (11800,6500,38),(13800,2900,45),(16500,8200,35),(17300,1400,65),
    (23500,700,100),(28200,4700,75),(28700,2400,55),(31500,850,130),
    (35000,450,180),(47000,1700,100)
]
for center, amplitude, width in peak_specs:
    raw += amplitude*np.exp(-0.5*((mz-center)/width)**2)

raw = np.clip(raw, 0, None)
spectrum = pd.DataFrame({"mz": mz, "intensity": raw})
spectrum.head()


## 5. (a) Raw spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(9,4.8))
ax.plot(mz, raw, linewidth=0.8)
ax.set(xlabel="m/z", ylabel="Intensity", title="(a) Raw spectrum")
ax.set_xlim(mz.min(), mz.max())
plt.tight_layout()
plt.show()


## 6. (b) Savitzky–Golay smoothing

In [ ]:
smoothed = smooth_spectrum(raw, window_length=41, polyorder=3)
spectrum["smoothed"] = smoothed

fig, ax = plt.subplots(figsize=(9,4.8))
ax.plot(mz, smoothed, linewidth=1.0)
ax.set(xlabel="m/z", ylabel="Intensity", title="(b) Smoothed spectrum")
ax.set_xlim(mz.min(), mz.max())
plt.tight_layout()
plt.show()


## 7. (c) Baseline correction

In [ ]:
baseline_corrected, baseline = baseline_correct(smoothed, degree=3)
spectrum["baseline"] = baseline
spectrum["baseline_corrected"] = baseline_corrected

fig, ax = plt.subplots(figsize=(9,4.8))
ax.plot(mz, baseline_corrected, linewidth=1.0)
ax.set(xlabel="m/z", ylabel="Intensity", title="(c) Smoothed + baseline corrected")
ax.set_xlim(mz.min(), mz.max())
plt.tight_layout()
plt.show()


## 8. (d) Peak detection

In [ ]:
peak_indices, peak_properties = detect_peaks(
    mz, baseline_corrected, prominence=120, distance=40
)
peaks = pd.DataFrame({
    "mz": mz[peak_indices],
    "intensity": baseline_corrected[peak_indices],
    "prominence": peak_properties["prominences"],
})
display(peaks)


In [ ]:
fig, ax = plt.subplots(figsize=(9,4.8))
ax.plot(mz, baseline_corrected, linewidth=1.0)
ax.scatter(
    mz[peak_indices], baseline_corrected[peak_indices],
    s=55, facecolors="none", edgecolors="black", linewidths=1.2
)
ax.set(xlabel="m/z", ylabel="Intensity", title="(d) Peak detection")
ax.set_xlim(mz.min(), mz.max())
plt.tight_layout()
plt.show()


## 9. Complete four-panel reproduction

This creates the same **processing sequence and panel arrangement** as Figure 2: raw, smoothed, baseline-corrected, and peak-detected spectra.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,8), sharex=True)

axes[0,0].plot(mz, raw, linewidth=0.8)
axes[0,0].set_title("(a) Raw spectrum")

axes[0,1].plot(mz, smoothed, linewidth=1.0)
axes[0,1].set_title("(b) Smoothed spectrum")

axes[1,0].plot(mz, baseline_corrected, linewidth=1.0)
axes[1,0].set_title("(c) Smoothed + baseline corrected")

axes[1,1].plot(mz, baseline_corrected, linewidth=1.0)
axes[1,1].scatter(
    mz[peak_indices], baseline_corrected[peak_indices],
    s=55, facecolors="none", edgecolors="black", linewidths=1.2
)
axes[1,1].set_title("(d) Peak detection")

for ax in axes.flat:
    ax.set_xlabel("m/z")
    ax.set_ylabel("Intensity")
    ax.set_xlim(mz.min(), mz.max())

plt.tight_layout()
plt.show()


## 10. Parameter sensitivity

Peak detection is strongly affected by `prominence` and `distance`. Try changing these values before analyzing real spectra.


In [ ]:
PROMINENCE = 250
DISTANCE = 60

idx, props = detect_peaks(
    mz, baseline_corrected,
    prominence=PROMINENCE, distance=DISTANCE
)

fig, ax = plt.subplots(figsize=(11,5))
ax.plot(mz, baseline_corrected, linewidth=1.0)
ax.scatter(
    mz[idx], baseline_corrected[idx],
    s=55, facecolors="none", edgecolors="black", linewidths=1.2
)
ax.set(
    xlabel="m/z",
    ylabel="Intensity",
    title=f"Peak detection: prominence={PROMINENCE}, distance={DISTANCE}"
)
plt.tight_layout()
plt.show()

print("Detected peaks:", len(idx))


## 11. Optional: load a real mzML spectrum with pyOpenMS

The referenced `basic_spectral_manipulation.ipynb` uses pyOpenMS to read mzML files. The function below extracts one MS1 spectrum and feeds it into exactly the same preprocessing pipeline.


In [ ]:
def load_ms1_spectrum(mzml_file, spectrum_index=0):
    import pyopenms as oms

    exp = oms.MSExperiment()
    oms.MzMLFile().load(str(mzml_file), exp)

    ms1 = [s for s in exp if s.getMSLevel() == 1]
    if not ms1:
        raise ValueError("No MS1 spectra found.")

    mz_values, intensity_values = ms1[spectrum_index].get_peaks()
    return np.asarray(mz_values), np.asarray(intensity_values)

# Example:
# mz_real, intensity_real = load_ms1_spectrum("your_file.mzML", 0)
# smoothed_real = smooth_spectrum(intensity_real, 41, 3)
# corrected_real, baseline_real = baseline_correct(smoothed_real, 3)
# idx_real, props_real = detect_peaks(mz_real, corrected_real, 120, 40)


## 12. Export detected peaks

In [ ]:
peaks.to_csv("example_detected_peaks.csv", index=False)
print("Saved example_detected_peaks.csv")


## 13. Summary

**Raw → Savitzky–Golay smoothing → polynomial baseline correction → `find_peaks`**

This workflow is intentionally simple and is useful for teaching the fundamentals of spectral preprocessing before moving to more sophisticated metabolomics pipelines such as centroiding, deisotoping, alignment, feature detection, MS/MS annotation, and molecular networking.
